<a href="https://colab.research.google.com/github/BlackAngel2108/Analytics-Portfolio/blob/main/A_B_%D1%82%D0%B5%D1%81%D1%82_(z_%D1%82%D0%B5%D1%81%D1%82).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from scipy.stats import chi2_contingency

In [2]:
# 1. Симулируем данные
np.random.seed(42)
n_users = 10000

data = pd.DataFrame({
    'user_id': range(n_users),
    'group': np.random.choice(['control', 'test'], size=n_users, p=[0.5, 0.5])
})

# Задаем вероятности конверсии: Control = 5%, Test = 5.8% (есть небольшой эффект)
def assign_conversion(row):
    if row['group'] == 'control':
        return np.random.binomial(1, 0.05)
    else:
        return np.random.binomial(1, 0.058)

data['converted'] = data.apply(assign_conversion, axis=1)

In [3]:
# 2. Сводка
summary = data.groupby('group')['converted'].agg(['count', 'sum'])
print(summary)

         count  sum
group              
control   5076  270
test      4924  292


In [4]:
# 3. Z-test для пропорций
control_converted = summary.loc['control', 'sum']
control_n = summary.loc['control', 'count']
test_converted = summary.loc['test', 'sum']
test_n = summary.loc['test', 'count']

counts = [control_converted, test_converted]
nobs = [control_n, test_n]

In [5]:
p_control = control_converted / control_n
p_test = test_converted / test_n
print(f"CR Control: {p_control:.3f}, CR Test: {p_test:.3f}")
print(f"Lift: {(p_test/p_control - 1)*100:.2f}%")


CR Control: 0.053, CR Test: 0.059
Lift: 11.49%


In [6]:
# Статистика
stat, pval = proportions_ztest(counts, nobs, alternative='two-sided')
print(f"Z-stat: {stat:.3f}, P-value: {pval:.4f}")

if pval < 0.05:
    print("Результат статистически значим. Новый алгоритм работает лучше.")
else:
    print("Статистически значимых отличий нет.")

# Доверительные интервалы
ci_low, ci_high = proportion_confint(test_converted, test_n, alpha=0.05, method='wilson')
print(f"95% CI для тестовой группы: [{ci_low:.3f}, {ci_high:.3f}]")

Z-stat: -1.326, P-value: 0.1847
Статистически значимых отличий нет.
95% CI для тестовой группы: [0.053, 0.066]


In [8]:
import math
from scipy.stats import norm

# Параметры
p = 0.05          # базовая конверсия 5%
n = 5000          # размер группы
alpha = 0.05      # уровень значимости
beta = 0.2        # 1 - мощность (power = 0.8)

z_alpha = norm.ppf(1 - alpha/2)  # 1.96
z_beta = norm.ppf(1 - beta)      # 0.84

# MDE в абсолютных пунктах
mde_abs = (z_alpha + z_beta) * math.sqrt(2 * p * (1 - p) / n)

# MDE в относительных процентах
mde_rel = (mde_abs / p) * 100

print(f"MDE (абсолютный): {mde_abs:.4f} ({mde_abs*100:.2f} п.п.)")
print(f"MDE (относительный): {mde_rel:.1f}%")

MDE (абсолютный): 0.0122 (1.22 п.п.)
MDE (относительный): 24.4%


In [9]:
# Для детектирования эффекта +0.8 п.п.
effect = 0.008  # 0.8 п.п.

n_required = ((z_alpha + z_beta) ** 2) * (2 * p * (1 - p)) / (effect ** 2)
print(f"Требуемый размер группы: {int(n_required)}")

Требуемый размер группы: 11650


# A/B тестирование: оценка нового алгоритма скоринга

## Цель
Оценить влияние нового алгоритма скоринга на конверсию в оформление кредитной карты.

## Дизайн эксперимента
- **Группы:** контроль (старый процесс) vs тест (новый алгоритм)
- **Метрика:** конверсия (0/1)
- **Размер:** по 5000 пользователей в группе
- **Метод:** двухвыборочный Z-тест для пропорций

## Результаты

| Группа | Клиенты | Конверсии | Конверсия |
|--------|---------|-----------|----------|
| control | 5000 | 250 | 5.00% |
| test | 5000 | 290 | 5.80% |

- **Lift:** +11.49%
- **Z-статистика:** -1.326
- **p-value:** 0.1847

## Вывод
p-value = 0.1847 > 0.05 → статистически значимых различий не обнаружено.

Несмотря на наблюдаемый рост конверсии, эффект оказался ниже минимального детектируемого.. Рекомендуется увеличить размер выборки для подтверждения эффекта.

**Ключевые метрики A/B теста:**  
- Lift (относительный прирост)  
- p-value (статистическая значимость)  
- MDE (минимальный детектируемый эффект)